In [53]:
import numpy as np
import pandas as pd
#from sklearn.preprocessing import OrdinalEncoder

In [54]:
# Directory Dataset (HD) Cleaning & Label Mapping
import pandas as pd
import numpy as np

df_hd = pd.read_csv("../../../data/raw/IPEDS/IPEDS_institutional_directory_2023.csv")

target_columns_map = {
    "UNITID": "unitid",            
    "INSTNM": "institution_name",   
    "CITY": "city",                 
    "STABBR": "state",              
    "OBEREG": "bea_region",         
    "SECTOR": "sector",             
    "ICLEVEL": "iclevel",           
    "CONTROL": "control",           
    "HLOFFER": "highest_offering",  
    "UGOFFER": "offers_undergrad",  
    "GROFFER": "offers_grad",       
    "DEGGRANT": "degree_granting"   
}

# Safeguard: handle case variations in raw IPEDS headers
col_map_case_insensitive = {col.upper(): col for col in df_hd.columns}
available_cols = [col_map_case_insensitive[col] for col in target_columns_map.keys() if col in col_map_case_insensitive]

df_hd_clean = df_hd[available_cols].copy()
rename_dict = {col_map_case_insensitive[orig]: clean for orig, clean in target_columns_map.items() if orig in col_map_case_insensitive}
df_hd_clean.rename(columns=rename_dict, inplace=True)


df_hd_clean = df_hd_clean[df_hd_clean["sector"] != 0].copy()

# Standardize Binary Indicators (1 = Yes, 2/Negative = 0)
hd_binary_cols = ["offers_undergrad", "offers_grad", "degree_granting"]
for col in hd_binary_cols:
    if col in df_hd_clean.columns:
        df_hd_clean[col] = df_hd_clean[col].apply(lambda x: 1 if x == 1 else 0)


# BEA Region Mapping
bea_map = {
    1: "New England",
    2: "Mid East",
    3: "Great Lakes",
    4: "Plains",
    5: "Southeast",
    6: "Southwest",
    7: "Rocky Mountains",
    8: "Far West",
    0: "Outlying Areas",
    9: "Outlying Areas"
}
df_hd_clean["bea_region"] = df_hd_clean["bea_region"].map(bea_map)

# Sector Mapping (9 active operational categories)
sector_map = {
    0: "Administrative unit",
    1: "Public, 4-year or above",
    2: "Private non-profit, 4-year or above",
    3: "Private for-profit, 4-year or above",
    4: "Public, 2-year",
    5: "Private non-profit, 2-year",
    6: "Private for-profit, 2-year",
    7: "Public, less-than-2-year",
    8: "Private non-profit, less-than-2-year",
    9: "Private for-profit, less-than-2-year"
}
df_hd_clean["sector"] = df_hd_clean["sector"].map(sector_map)

# Institutional Level (ICLEVEL) Mapping
iclevel_map = {
    1: "4-year or above",
    2: "2-year",
    3: "Less than 2-year"
}
df_hd_clean["iclevel"] = df_hd_clean["iclevel"].map(iclevel_map)

# Institutional Control Mapping
control_map = {
    1: "Public",
    2: "Private non-profit",
    3: "Private for-profit"
}
df_hd_clean["control"] = df_hd_clean["control"].map(control_map)

# Highest Offering Mapping
hloffer_map = {
    1: "Certificate (< 1 year)",
    2: "Certificate (1 to 2 years)",
    3: "Associate's degree",
    4: "Certificate (2 to 4 years)",
    5: "Bachelor's degree",
    6: "Postbaccalaureate certificate",
    7: "Master's degree",
    8: "Post-master's certificate",
    9: "Doctor's degree"
}
df_hd_clean["highest_offering"] = df_hd_clean["highest_offering"].map(hloffer_map)

# Save HD dataset
#df_hd_clean.to_csv("ipeds_clean_hd.csv", index=False)
print(f"Cell 1 Complete - Clean HD Dataset Shape: {df_hd_clean.shape}")

Cell 1 Complete - Clean HD Dataset Shape: (6092, 12)


In [55]:
# Institutional Characteristics (IC) Cleaning
df_ic = pd.read_csv("../../../data/raw/IPEDS/IPEDS_institutional_characteristics_2023.csv")

ic_columns_map = {
    "UNITID": "unitid",
    "PEO1ISTR": "offers_occupational_prog",
    "PEO2ISTR": "offers_academic_prog",
    "PEO3ISTR": "offers_continuing_ed",
    "PEO4ISTR": "offers_recreational_prog",
    "PEO5ISTR": "offers_remedial_instruction",
    "PEO6ISTR": "offers_secondary_prog",
    "CALSYS": "calendar_system",
    "FT_UG": "enrolls_ft_undergrad",
    "PT_UG": "enrolls_pt_undergrad",
    "DOCAC": "uses_clock_hours",
    "SLO5": "provides_job_placement",
    "SLO6": "provides_employment_services",
    "DISTNCED": "offers_distance_courses",
    "DSTNCED1": "offers_cert_under_12wks",
    "DSTNCED2": "offers_cert_12to52wks"
}

# Safeguard column name casing across raw headers
col_map_ic = {col.upper(): col for col in df_ic.columns}
available_ic_cols = [col_map_ic[col] for col in ic_columns_map.keys() if col in col_map_ic]

df_ic_clean = df_ic[available_ic_cols].copy()
rename_ic_dict = {col_map_ic[orig]: clean for orig, clean in ic_columns_map.items() if orig in col_map_ic}
df_ic_clean.rename(columns=rename_ic_dict, inplace=True)

# Recode All Indicator Features to Standard Binary (1 / 0)
# Keeping 0s so non-occupational providers serve as negative controls
indicator_cols = [c for c in df_ic_clean.columns if c not in ["unitid", "calendar_system"]]

for col in indicator_cols:
    df_ic_clean[col] = df_ic_clean[col].apply(lambda x: 1 if x == 1 else 0)

# Save IC dataset
#df_ic_clean.to_csv("ipeds_clean_ic.csv", index=False)
print(f"Cell 2 Complete - Clean IC Dataset Shape: {df_ic_clean.shape}")

Cell 2 Complete - Clean IC Dataset Shape: (6049, 15)


In [56]:
# Merging & Workforce Feature Creation

# 1. Merge HD and IC on unitid
df_merged = pd.merge(df_hd_clean, df_ic_clean, on="unitid", how="inner")

# Feature A: Sub-baccalaureate institution flag (2-year or less-than-2-year)
df_merged["is_sub_baccalaureate"] = df_merged["iclevel"].isin(["2-year", "Less than 2-year"]).astype(int)

# Feature B: Short-term occupational certificate provider flag
cert_cols = [c for c in ["offers_cert_under_12wks", "offers_cert_12to52wks"] if c in df_merged.columns]
if cert_cols:
    df_merged["is_short_term_cert_provider"] = (df_merged[cert_cols].sum(axis=1) > 0).astype(int)
else:
    df_merged["is_short_term_cert_provider"] = 0

# Feature C: Comprehensive career & employment support flag
if "provides_job_placement" in df_merged.columns and "provides_employment_services" in df_merged.columns:
    df_merged["has_career_support_infrastructure"] = (
        (df_merged["provides_job_placement"] == 1) & 
        (df_merged["provides_employment_services"] == 1)
    ).astype(int)

# 4. Save the updated df_merged back to your processed folder
df_merged.to_csv("../../../data/processed/IPEDS/IPEDS_dataset_cleaned.csv", index=False)

print(f"Cell 3 Complete - Final Dataset Shape: {df_merged.shape}")
df_merged.head()

Cell 3 Complete - Final Dataset Shape: (5978, 29)


,unitid,institution_name,city,state,bea_region,sector,iclevel,control,highest_offering,offers_undergrad,...,enrolls_ft_undergrad,enrolls_pt_undergrad,provides_job_placement,provides_employment_services,offers_distance_courses,offers_cert_under_12wks,offers_cert_12to52wks,is_sub_baccalaureate,is_short_term_cert_provider,has_career_support_infrastructure
0,100654,Alabama A & M University,Normal,AL,Southeast,"Public, 4-year or above",4-year or above,Public,Doctor's degree,1,...,1,1,1,1,0,1,1,0,1,1
1,100663,University of Alabama at Birmingham,Birmingham,AL,Southeast,"Public, 4-year or above",4-year or above,Public,Doctor's degree,1,...,1,1,1,1,0,1,1,0,1,1
2,100690,Amridge University,Montgomery,AL,Southeast,"Private non-profit, 4-year or above",4-year or above,Private non-profit,Doctor's degree,1,...,1,1,0,0,1,1,1,0,1,0
3,100706,University of Alabama in Huntsville,Huntsville,AL,Southeast,"Public, 4-year or above",4-year or above,Public,Doctor's degree,1,...,1,1,1,1,0,1,1,0,1,1
4,100724,Alabama State University,Montgomery,AL,Southeast,"Public, 4-year or above",4-year or above,Public,Doctor's degree,1,...,1,1,1,1,0,1,1,0,1,1


In [57]:
DATA_RAW = "../../../data/raw/IPEDS/"   # adjust to where the ic2023 files live

ay = pd.read_csv(DATA_RAW + "IPEDS_student_charges_tuition_ic2023.csv", encoding="latin-1")
py = pd.read_csv(DATA_RAW + "IPEDS_program_charges_ic2023.csv", encoding="latin-1",
                 dtype={f"CIPCODE{i}": str for i in range(1, 7)})
ay.columns = ay.columns.str.lower()
py.columns = py.columns.str.lower()

def clean_numeric(df, cols):
    """Coerce to numeric; IPEDS negative codes (-1, -2, -3) become NaN."""
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[df[c] < 0, c] = np.nan
    return df

In [58]:
ay_map = {
    "tuition1": "ay_tuition_indistrict", "tuition2": "ay_tuition_instate", "tuition3": "ay_tuition_outstate",
    "fee1": "ay_fee_indistrict", "fee2": "ay_fee_instate", "fee3": "ay_fee_outstate",
    "hrchg1": "ay_credithr_indistrict", "hrchg2": "ay_credithr_instate", "hrchg3": "ay_credithr_outstate",
    "chg1ay3": "ay_pub_tuitfee_indistrict", "chg2ay3": "ay_pub_tuitfee_instate",
    "chg3ay3": "ay_pub_tuitfee_outstate", "chg4ay3": "ay_books_supplies",
}
ay_sel = ay[["unitid", *ay_map]].rename(columns=ay_map)
ay_sel = clean_numeric(ay_sel, list(ay_map.values()))

In [59]:
# Institution-level PY fields (exist once per school)
# Removed chg5py3 / chg6py3 (on-campus room and board / other): ~99% NA
py_inst_map = {"prgmofr": "py_n_programs_offered", "lnayhr1": "py_acad_year_hours",
               "lnaywk1": "py_acad_year_weeks",
               "chg1py3": "py_tuitfee", "chg4py3": "py_books_supplies",
               "chg7py3": "py_offcampus_roomboard", "chg8py3": "py_offcampus_other",
               "chg9py3": "py_offcampus_family_other"}
py_inst = py[["unitid", *py_inst_map]].rename(columns=py_inst_map)
py_inst = clean_numeric(py_inst, list(py_inst_map.values()))

# Program-level: wide -> long across the 6 program slots
parts = []
for i in range(1, 7):
    cols = ["unitid", f"cipcode{i}", f"ciptuit{i}", f"cipsupp{i}",
            f"ciplgth{i}", f"prgmsr{i}", f"mthcmp{i}"]
    names = ["unitid", "cip_code", "prog_tuition_fees", "prog_supplies",
             "prog_length", "length_unit_code", "months_to_complete"]
    if i == 1:                      # weeks to complete exists for slot 1 only
        cols.append("wkcmp1")
        names.append("weeks_to_complete")
    p = py[cols].copy()
    p.columns = names
    p["program_slot"] = i
    parts.append(p)

prog = pd.concat(parts, ignore_index=True)
prog = prog[pd.to_numeric(prog.cip_code, errors="coerce") > 0].copy()   # drop empty (-2) slots
prog = clean_numeric(prog, ["prog_tuition_fees", "prog_supplies", "prog_length",
                            "months_to_complete", "weeks_to_complete"])
prog["cip_2digit"] = prog.cip_code.str[:2]
prog["length_unit"] = (pd.to_numeric(prog.length_unit_code, errors="coerce")
                         .map({1: "credit_hours", 2: "contact_hours"}))  # verify codes in dictionary

# Slot 1 costs are reported in CHG1PY3 / CHG4PY3 when CIPTUIT1 / CIPSUPP1 are not applicable
t = py_inst.set_index("unitid")
s1 = prog.program_slot.eq(1)
prog.loc[s1, "prog_tuition_fees"] = prog.loc[s1, "prog_tuition_fees"].fillna(prog.loc[s1, "unitid"].map(t["py_tuitfee"]))
prog.loc[s1, "prog_supplies"] = prog.loc[s1, "prog_supplies"].fillna(prog.loc[s1, "unitid"].map(t["py_books_supplies"]))

In [60]:
inst = (df_merged.merge(ay_sel, on="unitid", how="left", validate="1:1")
                 .merge(py_inst, on="unitid", how="left", validate="1:1"))
inst["price_reporting"] = np.select(
    [inst.unitid.isin(ay.unitid), inst.unitid.isin(py.unitid)],
    ["academic_year", "program_year"], default="none")

prog_df = prog.merge(df_merged, on="unitid", how="left", validate="m:1")

assert inst.unitid.is_unique and len(inst) == len(df_merged)
print(inst.price_reporting.value_counts())
print(prog_df.shape, prog_df.unitid.nunique())

price_reporting
academic_year    3825
program_year     2141
none               12
Name: count, dtype: int64
(7953, 39) 2141


In [61]:
prog_df.head()

,unitid,cip_code,prog_tuition_fees,prog_supplies,prog_length,length_unit_code,months_to_complete,weeks_to_complete,program_slot,cip_2digit,...,enrolls_ft_undergrad,enrolls_pt_undergrad,provides_job_placement,provides_employment_services,offers_distance_courses,offers_cert_under_12wks,offers_cert_12to52wks,is_sub_baccalaureate,is_short_term_cert_provider,has_career_support_infrastructure
0,101277,12.0401,15150.0,3258.0,1500.0,1,12.0,47.0,1,12,...,1,0,0,0,0,0,0,1,0,0
1,102711,48.0508,4736.0,8048.0,1080.0,1,10.0,38.0,1,48,...,1,1,0,0,0,0,0,1,0,0
2,103501,51.0801,16545.0,NaN,46.0,2,10.0,40.0,1,51,...,1,0,0,0,0,0,0,1,0,0
3,103741,12.0401,19260.0,2750.0,1600.0,1,12.0,46.0,1,12,...,1,1,0,0,0,0,0,1,0,0
4,103893,51.0001,15711.0,0.0,60.0,2,11.0,42.0,1,51,...,1,0,0,0,0,1,0,1,1,0


In [62]:
# One table: every school from your clean IPEDS dataset (+ AY/PY costs),
# expanded to one row per program where program data exists
final = inst.merge(prog, on="unitid", how="left", validate="1:m")

# Derived program features (NaN where there is no program row)
final["total_prog_cost"] = final[["prog_tuition_fees", "prog_supplies"]].sum(axis=1, min_count=1)
final["cost_per_hour"] = final.total_prog_cost / final.prog_length
# Uses months (0% NA among program rows) instead of weeks (slot 1 only)
final["hours_per_month"] = final.prog_length / final.months_to_complete.replace(0, np.nan)

# Missing-value indicators (no imputation across AY / PY reporters)
for c in ["ay_pub_tuitfee_instate", "ay_books_supplies", "py_books_supplies", "prog_supplies"]:
    final[f"{c}_missing"] = final[c].isna().astype(int)

# Checks
assert final.unitid.isin(df_merged.unitid).all()
assert final.unitid.nunique() == len(df_merged)      # no schools lost
print(final.shape)
print(final.price_reporting.value_counts())

# NA check by reporting type
new_cols = [c for c in final.columns if c not in df_merged.columns]
na = pd.DataFrame({
    "na_all_%": final[new_cols].isna().mean() * 100,
    "na_AY_%":  final.loc[final.price_reporting == "academic_year", new_cols].isna().mean() * 100,
    "na_PY_%":  final.loc[final.price_reporting == "program_year",  new_cols].isna().mean() * 100,
}).round(1)
display(na)

# Optional: program-level only (drops AY schools, which have no program rows)
# final = final[final.cip_code.notna()]

final.to_csv("../../../data/processed/IPEDS/IPEDS_dataset_cleaned.csv", index=False)

(11790, 68)
price_reporting
program_year     7953
academic_year    3825
none               12
Name: count, dtype: int64


,na_all_%,na_AY_%,na_PY_%
ay_tuition_indistrict,70.0,7.5,100.0
ay_tuition_instate,70.0,7.5,100.0
ay_tuition_outstate,70.0,7.5,100.0
ay_fee_indistrict,70.0,7.5,100.0
ay_fee_instate,70.0,7.5,100.0
ay_fee_outstate,70.0,7.5,100.0
ay_credithr_indistrict,73.1,17.0,100.0
ay_credithr_instate,73.1,17.0,100.0
ay_credithr_outstate,72.9,16.4,100.0
ay_pub_tuitfee_indistrict,71.0,10.6,100.0


In [63]:
new_cols = [c for c in final.columns if c not in df_merged.columns]
na = pd.DataFrame({
    "na_all_%": final[new_cols].isna().mean() * 100,
    "na_AY_%":  final.loc[final.price_reporting == "academic_year", new_cols].isna().mean() * 100,
    "na_PY_%":  final.loc[final.price_reporting == "program_year",  new_cols].isna().mean() * 100,
}).round(1)
display(na)

prog_rows = final[final.cip_code.notna()]
display(prog_rows[["prog_tuition_fees", "prog_supplies", "prog_length",
                   "weeks_to_complete", "total_prog_cost"]].isna().mean().mul(100).round(1))

,na_all_%,na_AY_%,na_PY_%
ay_tuition_indistrict,70.0,7.5,100.0
ay_tuition_instate,70.0,7.5,100.0
ay_tuition_outstate,70.0,7.5,100.0
ay_fee_indistrict,70.0,7.5,100.0
ay_fee_instate,70.0,7.5,100.0
ay_fee_outstate,70.0,7.5,100.0
ay_credithr_indistrict,73.1,17.0,100.0
ay_credithr_instate,73.1,17.0,100.0
ay_credithr_outstate,72.9,16.4,100.0
ay_pub_tuitfee_indistrict,71.0,10.6,100.0


prog_tuition_fees     0.2
prog_supplies         9.7
prog_length           0.0
weeks_to_complete    73.7
total_prog_cost       0.2
dtype: float64

In [64]:
feature_summary = pd.DataFrame({
    "Data Type": df_merged.dtypes,
    "Unique Values": df_merged.nunique(),
    "Sample Value": df_merged.iloc[0]
})

feature_summary

,Data Type,Unique Values,Sample Value
unitid,int64,5978,100654
institution_name,str,5880,Alabama A & M University
city,str,2273,Normal
state,str,59,AL
bea_region,str,9,Southeast
sector,str,9,"Public, 4-year or above"
iclevel,str,3,4-year or above
control,str,3,Public
highest_offering,str,9,Doctor's degree
offers_undergrad,int64,2,1
